# Convolutional Neural Networks

Import of the used packages and methods

In [1]:
import numpy as np
import tensorflow as tf
import tsfel
from tensorflow.keras import layers, models
import pandas as pd
import datetime

I0000 00:00:1787600304.392747  489319 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787600304.393201  489319 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787600304.425416  489319 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787600305.073955  489319 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

The Tensorboard extension was used to visualize the history of training

In [2]:
%load_ext tensorboard

# Split Train/Test in 5 folds
splitted to Dicts of:
- meta_tr (train)
- meta_te (test)

and each group will appear exactly once in the test set across all folds

In [3]:
from sklearn.model_selection import GroupKFold
from auswertung_features.build_features import split_into_label_blocks, load_session

meta = pd.read_csv("../auswertung_features/meta.csv")
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
meta_tr: dict[int, pd.DataFrame] = {}
meta_te: dict[int, pd.DataFrame] = {}

dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "block_id":[]}
for i, r in meta.iterrows():
        csv_path = r["rel_path"]
        df_session = load_session(csv_path)

        #Error Handling für leere dfs
        if df_session is None:
            continue

        blocks = split_into_label_blocks(df_session)
        for block in blocks:
            meta_blocks["subject_id"].append(r["subject_id"])
            meta_blocks["hand"].append(r["hand"])
            meta_blocks["block_id"].append(len(dict_blocks))
            dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(meta_blocks, groups=meta_blocks["subject_id"])):
    meta_tr[fold] = meta_blocks.iloc[tr_idx].reset_index(drop=True)
    meta_te[fold] = meta_blocks.iloc[te_idx].reset_index(drop=True)


[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty


## Convert dataset to match TensorFlow Conventions

Input needs to be in shape (batch, steps, features). My own blocks (ndarray) are in the shape of (sample_length, sensor_channels). The steps are the sequence length which should be learned.

In [4]:
def build_xy_from_blocks(meta_df, dict_blocks):
    X_list, y_list = [], []
    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        X_list.append(np.asarray(Xb, dtype=np.float32))
        y_list.append(int(yb))   # bleibt 0..4
    return X_list, np.asarray(y_list, dtype=np.int32)


In [5]:
def make_dataset(X_list, y, batch_size=32, training=True):
    X_list = [np.asarray(x, dtype=np.float32) for x in X_list]
    y = np.asarray(y, dtype=np.int32)

    def gen():
        for x_i, y_i in zip(X_list, y):
            yield x_i, y_i

    output_signature = (
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    if training:
        ds = ds.shuffle(buffer_size=len(y), reshuffle_each_iteration=True)

    ds = ds.padded_batch(
        batch_size=batch_size,
        padded_shapes=([None, 4], []),
        padding_values=(0.0, 0)
    ).prefetch(tf.data.AUTOTUNE)

    return ds

In [6]:
def seq_to_windows(x,y, win_len=256, stride=128):
    """
    x: (T, 4) float32
    y: int (Klasse)
    returns: Xw (n_windows, win_len, 4), yw (n_windows,)
    """
    T = x.shape[0]
    if T < win_len:
        return np.empty((0, win_len, x.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.int32)

    starts = range(0, T - win_len + 1, stride)
    Xw = np.stack([x[s:s+win_len] for s in starts]).astype(np.float32)
    yw = np.full((Xw.shape[0],), int(y), dtype=np.int32)
    return Xw, yw

In [7]:
def build_windowed_xy(meta_df, dict_blocks, win_len=256, stride=128):
    X_all = []
    y_all = []

    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        Xb = np.asarray(Xb, dtype=np.float32)
        yb = int(yb)  # 0..4

        Xw, yw = seq_to_windows(Xb, yb, win_len=win_len, stride=stride)
        if len(yw) == 0:
            continue

        X_all.append(Xw)
        y_all.append(yw)

    X_all = np.concatenate(X_all, axis=0)  # (N, win_len, 4)
    y_all = np.concatenate(y_all, axis=0)  # (N,)
    return X_all, y_all


# Shaping the Batch
This defines how the dataset should be formed. This one gives the whole sequence as batches to train

In [ ]:
fold = 0
str_hand = "r"

mask_tr = meta_tr[fold]["hand"]==str_hand
mask_te = meta_te[fold]["hand"]==str_hand
X_tr, y_tr = build_xy_from_blocks(meta_tr[fold][mask_tr], dict_blocks)
X_te, y_te = build_xy_from_blocks(meta_te[fold][mask_te], dict_blocks)

ds_tr = make_dataset(X_tr, y_tr, batch_size=32, training=True).repeat(5)
ds_te = make_dataset(X_te, y_te, batch_size=32, training=False).repeat(5)

# Shape-Check
for xb, yb in ds_tr.take(1):
    print("xb:", xb.shape, "yb:", yb.shape)  # xb z.B. (32, 1190, 4)


This part defines the sequences into windows, which also makes every window the same length. The ends will be cut off

In [8]:
fold = 0
win_len, stride = 250, 50 # stride is like a step here
str_hand = "r"

mask_tr = meta_tr[fold]["hand"]==str_hand
mask_te = meta_te[fold]["hand"]==str_hand
X_tr_w, y_tr_w = build_windowed_xy(meta_tr[fold][mask_tr], dict_blocks, win_len, stride)
X_te_w, y_te_w = build_windowed_xy(meta_te[fold][mask_te], dict_blocks, win_len, stride)

print(X_tr_w.shape, y_tr_w.shape)  # z.B# . (N, 256, 4), (N,)

ds_tr = tf.data.Dataset.from_tensor_slices((X_tr_w, y_tr_w)) \
    .shuffle(len(y_tr_w)) \
    .batch(64) \
    .prefetch(tf.data.AUTOTUNE)

ds_te = tf.data.Dataset.from_tensor_slices((X_te_w, y_te_w)) \
    .batch(64) \
    .prefetch(tf.data.AUTOTUNE)

(6872, 250, 4) (6872,)


E0000 00:00:1787600314.746409  489319 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Building a feature Table
Building a feature table to feed it into the DNN. To see if it helps

In [ ]:
from auswertung_features.parameterraum import DataConfig
from auswertung_features.build_features import build_feature_table
from auswertung_features.parameter_suche import data_cfg_key
import os
# NOT DONE
tsfel_cfg = tsfel.get_features_by_domain(json_path="../auswertung_features/tsfel_conf.json")
dcfg = DataConfig(
    trim=0,
    min_len=win_len,
    win=win_len,
    step=stride,
    feature_set_name= "tsfel"
)
feature_cache: dict[str, pd.DataFrame] = {}

k = data_cfg_key(dcfg)
if k not in feature_cache:
    if os.path.isfile(f"../auswertung_features/cache/feature_{k}.csv"):
        feature_cache[k] = pd.read_csv(f"../auswertung_features/cache/feature_{k}.csv", index_col = 0, dtype={"subject_id":str})
    else:
        feature_cache[k] = build_feature_table(meta=meta, data_cfg=dcfg, tsfel_cfg=tsfel_cfg)
        feature_cache[k].to_csv(f"../auswertung_features/cache/feature_{k}.csv")

print()

# First Test of CNN and Gru
Down here is a first idea or test of the 1d CNN. If you input the whole sequence as train set, you get a max accuracy on the validation set on around 0.5/0.55


In [10]:
import keras
initializer = tf.keras.initializers.HeNormal(seed = 0)

wd = 1e-3

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(16, 7, padding="same", activation="leaky_relu", kernel_initializer=initializer,
                  kernel_regularizer=keras.regularizers.l2(wd))(inputs)
#x = layers.MaxPooling1D()(x)
x = layers.SpatialDropout1D(0.2)(x)
x = layers.Conv1D(32, 5, padding="same", activation="leaky_relu", kernel_regularizer=keras.regularizers.l2(wd))(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(5, activation="softmax", kernel_regularizer=keras.regularizers.l2(wd))(x)

model = models.Model(inputs, outputs)
optimizer = tf.keras.optimizers.Adadelta(learning_rate=1.0)
loss = keras.losses.SparseCategoricalCrossentropy()
model.compile(optimizer=optimizer,
              loss=loss,
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=1000, callbacks=[tensorboard_callback])



KeyboardInterrupt: 

This one is the first test to implement a 1d CNN --> GRU (RNN). It is pretty bad

In [12]:
initializer = tf.keras.initializers.HeNormal(seed = 0)

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(32, 7, padding="same", activation="leaky_relu",
                  kernel_initializer=initializer)(inputs)
x = layers.MaxPooling1D()(x)
x = layers.GRU(16, activation="leaky_relu")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=1000, callbacks=[tensorboard_callback])


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, None, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, None, 32)       │           928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, None, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16)             │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,413 (13.33 KB)

 Trainable params: 3,413 (13.33 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2053 - loss: 30.2924 - val_accuracy: 0.2017 - val_loss: 28.3175
Epoch 2/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2015 - loss: 21.1741 - val_accuracy: 0.2013 - val_loss: 21.6785
Epoch 3/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2043 - loss: 15.4532 - val_accuracy: 0.1941 - val_loss: 16.9228
Epoch 4/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.2058 - loss: 11.0748 - val_accuracy: 0.2013 - val_loss: 12.0358
Epoch 5/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2050 - loss: 7.6955 - val_accuracy: 0.2024 - val_loss: 8.8935
Epoch 6/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2139 - loss: 5.6441 - val_accuracy: 0.1949 - val_loss: 6.7629
Epoch 7/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2159 - loss: 4.2787 - val_accuracy: 0.1983 - val_loss: 5.3164
Epoch 8/1000
108/108 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2

Trying a standard model which was described by Kiranyaz et al. (2021) in 1D convolutional neural networks and applications: A survey

In [ ]:
import keras
initializer = tf.keras.initializers.HeNormal(seed = 0)
inputs = layers.Input(shape=(None, 4))

x = layers.Conv1D(20, 15, padding="same", activation="leaky_relu", kernel_initializer=initializer)(inputs)
x = layers.Conv1D(40, 24, padding="same", activation="leaky_relu")(x)
x = layers.Conv1D(60, 30, padding="same", activation="leaky_relu")(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(10, activation="softmax")(x)
outputs = layers.Dense(5, activation="softmax")(x)

loss = keras.losses.CategoricalGeneralizedCrossEntropy(q = 0.1)
model = models.Model(inputs, outputs)
optimizer = tf.keras.optimizers.Adadelta(learning_rate=1.0)
model.compile(optimizer=optimizer,
              loss=loss,
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=2000, callbacks=[tensorboard_callback])

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, None, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, None, 20)       │         1,220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, None, 40)       │        19,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, None, 60)       │        72,060 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 60)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │           610 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │            55 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,185 (364.00 KB)

 Trainable params: 93,185 (364.00 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.2055 - loss: 1.4991 - val_accuracy: 0.2069 - val_loss: 1.4938
Epoch 2/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2117 - loss: 1.4901 - val_accuracy: 0.2069 - val_loss: 1.4881
Epoch 3/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2117 - loss: 1.4871 - val_accuracy: 0.2069 - val_loss: 1.4866
Epoch 4/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2117 - loss: 1.4864 - val_accuracy: 0.2069 - val_loss: 1.4863
Epoch 5/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2117 - loss: 1.4862 - val_accuracy: 0.2069 - val_loss: 1.4862
Epoch 6/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2117 - loss: 1.4862 - val_accuracy: 0.2069 - val_loss: 1.4862
Epoch 7/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.2117 - loss: 1.4861 - val_accuracy: 0.2069 - val_loss: 1.4862
Epoch 8/2000
108/108 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2085 - lo

# PET Infrastructure build for classification
Right now it is an AI draft based on Lin et al. on A parallel and efficient transformer deep learning network for continuous estimation of hand kinematics from electromyographic signals

In [ ]:
import tensorflow as tf

class ExternalAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, s=64, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.s = s
        self.dropout = dropout

        self.wk = tf.keras.layers.Dense(s, use_bias=False)       # d -> S
        self.wv = tf.keras.layers.Dense(d_model, use_bias=False) # S -> d
        self.drop = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # x: (B,T,d)
        a = self.wk(x)                            # (B,T,S)
        a = tf.nn.softmax(a, axis=-1)             # softmax over S (memory slots)
        a = self.drop(a, training=training)
        y = self.wv(a)                            # (B,T,d)
        return y


In [ ]:
class PETBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, s=64, ffn_mult=2, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ea = ExternalAttention(d_model, s=s, dropout=dropout)

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_model * ffn_mult, activation="relu"),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(d_model),
        ])
        self.drop = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # Pre-Norm
        h = self.norm(x)

        # parallel branches
        attn_out = self.ea(h, training=training)
        ffn_out  = self.ffn(h, training=training)

        y = attn_out + ffn_out
        y = self.drop(y, training=training)

        # residual
        return x + y


In [ ]:
def build_pet_lite_classifier(
    win_len,
    n_ch=4,
    n_classes=5,
    d_model=64,
    n_blocks=2,
    s=64,
    dropout=0.2,
    conv_stride=2
):
    inp = tf.keras.Input(shape=(win_len, n_ch), name="emg")

    # Optional: Normalization layer (adapt() separat auf Train-Dataset!)
    # norm = tf.keras.layers.Normalization(axis=-1)
    # x = norm(inp)
    x = inp

    # Conv frontend (patch/downsample): T -> ceil(T/stride)
    x = tf.keras.layers.Conv1D(d_model, kernel_size=9, strides=conv_stride,
                               padding="same", activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    # PET blocks
    for i in range(n_blocks):
        x = PETBlock(d_model=d_model, s=s, ffn_mult=2, dropout=dropout, name=f"pet_{i}")(x)

    # Pooling over time (sequence-to-vector)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    out = tf.keras.layers.Dense(n_classes, name="logits")(x)

    model = tf.keras.Model(inp, out, name="pet_lite_emg")
    return model


In [ ]:
model = build_pet_lite_classifier(win_len=250, n_ch=4, n_classes=5,
                                  d_model=64, n_blocks=2, s=64, dropout=0.2)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)
model.summary()
